# 06 — Final Training, Evaluation & Export (Stage 2)

**Purpose:** Train the best configuration identified from ablation studies
(notebooks 04-05), run comprehensive evaluation, and export the model
for deployment.

**Outputs:**
- Final checkpoint (`.pth`)
- ONNX export (`.onnx`)
- TorchScript export (`.pt`)
- Comprehensive evaluation report (JSON + plots)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q yacs tqdm opencv-python-headless tensorboard onnx onnxruntime

import os, sys
REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

In [ ]:
import torch
import torch.nn as nn
from lib.config import cfg
from lib.models import get_net
from lib.utils.drive_dataset import (
    ensure_local_dataset_from_drive,
    find_raw_bdd_root,
    resolve_bdd_images_100k_dir,
    resolve_bdd_labels_100k_dir,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Default export target = best row. Flip CONFIG for ablation exports.
CONFIG = 'YOLOPv2-best-row'
yaml_map = {
    'YOLOPv2-best-row':   os.path.join(REPO_ROOT, 'configs', 'yolopv2_best_row.yaml'),
    'YOLOPv2-focal-only': os.path.join(REPO_ROOT, 'configs', 'yolopv2_focal_only_ablation.yaml'),
    'YOLOP':              os.path.join(REPO_ROOT, 'configs', 'yolop_vehicle_lane_baseline.yaml'),
}

cfg.defrost()
cfg.merge_from_file(yaml_map[CONFIG])

ECOCAR_ROOT = '/content/drive/MyDrive/EcoCAR'
DATASET_ROOT = ensure_local_dataset_from_drive('bdd100k_vehicle5', ECOCAR_ROOT)
RAW_BDD_ROOT = find_raw_bdd_root(ECOCAR_ROOT)
BDD_IMAGES = resolve_bdd_images_100k_dir(RAW_BDD_ROOT)
BDD_LABELS = resolve_bdd_labels_100k_dir(RAW_BDD_ROOT)
cfg.DATASET.ROOT = DATASET_ROOT
cfg.DATASET.DATAROOT = BDD_IMAGES
cfg.DATASET.LABELROOT = BDD_LABELS
cfg.DATASET.LANEROOT = os.path.join(DATASET_ROOT, 'masks')
cfg.freeze()

# Load best checkpoint for the selected config.
model = get_net(cfg).to(device)
model.gr = 1.0
model.nc = 5
model.names = cfg.MODEL.VEHICLE_CLASSES

ckpt_path = os.path.join(cfg.DRIVE.CHECKPOINT_DIR, 'best.pth')
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt['state_dict'])
model.eval()
print(f'[{CONFIG}] loaded checkpoint from epoch {ckpt.get("epoch", "?")}')


class YOLOPv2DemoWrapper(nn.Module):
    """Thin `nn.Module` that exposes the YOLOPv2 public-demo output
    contract so exported artifacts are byte-compatible with the
    external_repos/YOLOPv2/demo.py consumer side.

    Returns `(det_out, lane_prob_half_res_1ch)` via `model.predict(x)`.
    """

    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, x):
        return self.m.predict(x)


export_model = YOLOPv2DemoWrapper(model).to(device).eval()

In [ ]:
# ── ONNX Export — uses predict() wrapper (YOLOPv2 demo contract) ──
# Output contract (matches external_repos/YOLOPv2/demo.py):
#   det_out    — training-mode detection output (list of 3 scales)
#   lane_prob  — 1-channel sigmoid at H/2 × W/2
export_dir = os.path.join(cfg.DRIVE.CHECKPOINT_DIR, 'exports')
os.makedirs(export_dir, exist_ok=True)

dummy_input = torch.randn(1, 3, 640, 640).to(device)

onnx_path = os.path.join(export_dir, f'{CONFIG.lower()}.onnx')
torch.onnx.export(
    export_model, dummy_input, onnx_path,
    opset_version=17,                            # modern opset (GroupNorm, LayerNorm native)
    input_names=['images'],
    output_names=['det_out', 'lane_prob'],
    dynamic_axes={
        'images':    {0: 'batch'},
        'det_out':   {0: 'batch'},
        'lane_prob': {0: 'batch'},
    },
)
print(f'ONNX exported to {onnx_path}')

import onnx
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print('ONNX model validation passed.')

In [ ]:
# ── TorchScript Export — also uses the demo-contract wrapper ──
ts_path = os.path.join(export_dir, f'{CONFIG.lower()}.pt')
traced = torch.jit.trace(export_model, dummy_input)
traced.save(ts_path)
print(f'TorchScript exported to {ts_path}')

loaded = torch.jit.load(ts_path, map_location=device)
with torch.no_grad():
    out = loaded(dummy_input)
det_tuple, lane_prob = out
lane_shape = tuple(lane_prob.shape)
print(f'TorchScript check: lane_prob shape = {lane_shape}  (expect 1, 1, 320, 320 at 640×640 input)')

In [ ]:
# ── Model size summary ──
param_count = sum(p.numel() for p in model.parameters())
model_size_mb = os.path.getsize(ckpt_path) / 1024 / 1024
onnx_size_mb = os.path.getsize(onnx_path) / 1024 / 1024

print(f'Parameters:     {param_count/1e6:.2f}M')
print(f'Checkpoint:     {model_size_mb:.1f} MB')
print(f'ONNX:           {onnx_size_mb:.1f} MB')
print(f'Export dir:     {export_dir}')